# 00 — Kaggle kurulumu

**Bir kez yapılacak ayarlar** (sağdaki panel / üst menü):
- **Accelerator:** GPU T4 x2
- **Internet:** On
- **Environment:** Pin to original environment
- **Add-ons → Secrets:** `GH_TOKEN` (GitHub token'ı) ekli ve bu notebook için işaretli

Sonra **Run All**. Her yeni notebook'un başına ilk iki kod hücresini (Kurulum + Veri) kopyalayın.

In [ ]:
# 1) KURULUM — her oturumda ilk bu hücre çalışır
import os, subprocess
from kaggle_secrets import UserSecretsClient

REPO = "https://github.com/traldo44/kalori-tahmini.git"
GIT_NAME, GIT_EMAIL = "Ad Soyad", "eposta@ornek.com"   # ← kendi bilgilerinizi yazın
OWNER = "ad_soyad"                                      # ← results.csv'deki owner sütunu

os.environ["GH_TOKEN"] = UserSecretsClient().get_secret("GH_TOKEN")
# Token'ı hiçbir dosyaya yazmadan git'e verir; yalnızca bu oturumun hafızasında durur.
subprocess.run(["git", "config", "--global", "credential.helper",
                '!f() { echo username=x-access-token; echo "password=$GH_TOKEN"; }; f'], check=True)

os.chdir("/kaggle/working")
if os.path.isdir("kalori-tahmini"):
    subprocess.run(["git", "-C", "kalori-tahmini", "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO], check=True)
os.chdir("/kaggle/working/kalori-tahmini")
subprocess.run(["git", "config", "user.name", GIT_NAME], check=True)
subprocess.run(["git", "config", "user.email", GIT_EMAIL], check=True)

os.environ["KALORI_DATA"] = "/tmp/kalori-data"   # veri çıktı klasörüne (/kaggle/working) girmesin
os.environ["KALORI_OWNER"] = OWNER
print("Repo hazır →", subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
# 2) VERİ — Nutrition5k (~1,4 GB) + Food-101 25 sınıflık alt küme. Aynı oturumda tekrar çalışırsa atlar.
!python scripts/download_nutrition5k.py --out /tmp/kalori-data/nutrition5k
!python scripts/get_food101.py fetch --out /tmp/kalori-data


## Kontroller (yalnızca ilk seferde)
Hepsi ✓ ise Adım 1 tamam.

In [ ]:
import torch, timm, sklearn
gpu = torch.cuda.is_available()
print("GPU:", "✓ " + torch.cuda.get_device_name(0) if gpu else "✗ YOK → Settings → Accelerator → GPU T4 x2")
print("torch", torch.__version__, "| timm", timm.__version__, "| sklearn", sklearn.__version__)

In [ ]:
# Testler: hepsi 'passed' olmalı
!python -m pytest -q || (pip install -q pytest && python -m pytest -q)

In [ ]:
# GitHub yazma yetkisi: 'Everything up-to-date' görmelisiniz. 403 / Authentication failed → token'a Contents: Read and write verin.
!git push --dry-run
# Checkpoint deposuna erişim (şimdilik boş liste normal)
!python scripts/share_checkpoint.py list && echo "checkpoint erişimi ✓"